In [ ]:
import mne
import matplotlib.pyplot as plt
import numpy as np
from mne.stats import permutation_cluster_test
from mne.stats import ttest_ind_no_p
import pandas as pd

%matplotlib qt5
 

import os
import pandas as pd
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import mne
import PyQt5
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

from mne.stats import permutation_cluster_test as pcluster_test

In [ ]:
# вот этот код (+следующая ячейка), если нет заранее заготовленного tfr

selected_channels = ["Fz", "FCz", "Cz"]

all_epochs = [] 
data_folder = r"/Users/alexmedvedev/Desktop/Vishka/exp_epochs"
for file in os.listdir(data_folder):
    if file.endswith(".fif"):
        filepath = os.path.join(data_folder, file)
        print(f"Loading {filepath}")
        epochs = mne.read_epochs(filepath, preload=True)
        epochs.info['bads'] = []
        epochs.pick_channels(selected_channels)
        all_epochs.append(epochs)

epochs_combined = mne.concatenate_epochs(all_epochs)
epochs_combined

In [ ]:
freqs = np.arange(1, 99)  # or 45
vmin, vmax = -1, 3  
baseline = (-0.5, 0) # normal: (-1, -0.2)
cnorm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)  
tmin, tmax = -0.5, 1 # or 1, 3
kwargs = dict(n_permutations=500, step_down_p=0.05, seed=1, buffer_size=None, out_type="mask")  

tfr = epochs_combined.compute_tfr(
    method="multitaper",
    picks = roi,
    freqs=freqs,
    n_cycles=freqs / 2,
    use_fft=True,
    return_itc=False,
    average=False,
    decim=3, #6 or 3
    time_bandwidth=4.0  
)

tfr.apply_baseline(baseline=baseline, mode="percent")
tfr.crop(tmin=tmin, tmax=tmax)
tfr

In [ ]:
# или этот код, если есть сохраненный tfr со всеми каналами

tfr_path = Path('какой-то путь')
tfr = mne.time_frequency.read_tfrs(tfr_path)
selected_channels

In [ ]:
channel_indices = mne.pick_channels(tfr.ch_names, include=selected_channels)

In [ ]:
event_id_pairs = [("5-6", "7-8"), ("5-6", "9-10"), ("5-6", "11-12"),
                  ("7-8", "9-10"), ("7-8", "11-12"), ("9-10", "11-12")]

In [ ]:
kwargs = dict(n_permutations=500, tail=0, threshold=3.0, out_type="mask", n_jobs=1, stat_fun=ttest_ind_no_p)

In [ ]:
results = []

for event1, event2 in event_id_pairs:

    tfr_ev1 = tfr[event1].average()
    tfr_ev2 = tfr[event2].average()
    

    tfr_ev1.apply_baseline(baseline=(-0.5, 0), mode='zscore') 
    tfr_ev2.apply_baseline(baseline=(-0.5, 0), mode='zscore')


    print(f"Event pair: {event1} vs {event2}")
    print(f"tfr_ev1.data shape: {tfr_ev1.data.shape}")
    print(f"tfr_ev2.data shape: {tfr_ev2.data.shape}")


    fig, axes = plt.subplots(
        2, len(selected_channels) + 1,  
        figsize=(12, 8),
        gridspec_kw={"height_ratios": [1, 2], "width_ratios": [10] * len(selected_channels) + [1]}
    )

    for idx, (ch_idx, ax_mask, ax_diff) in enumerate(zip(channel_indices, axes[0, :-1], axes[1, :-1])):

        data1 = tfr_ev1.data[ch_idx, :, :] 
        data2 = tfr_ev2.data[ch_idx, :, :]  
        print(f"Channel {tfr.ch_names[ch_idx]} data shape: {data1.shape}")


        t_obs, clusters, cluster_p_values, H0 = permutation_cluster_test(
            [data1, data2],
            **kwargs
        )


        print(f"Number of clusters: {len(clusters)}")
        if clusters:
            for i, cluster in enumerate(clusters):
                print(f"Cluster {i} type: {type(cluster)}, Content: {cluster}, P-value: {cluster_p_values[i]}")


        cluster_mask = np.zeros(data1.shape, dtype=bool)  
        if clusters:
            for i, cluster in enumerate(clusters):
                if cluster_p_values[i] <= 0.05:
                    if isinstance(cluster, np.ndarray):  
                        cluster_mask = np.logical_or(cluster_mask, cluster)
                    else:  
                        mask = np.zeros(data1.shape, dtype=bool)
                        if len(cluster) == 1:  
                            
                            freq_slice = cluster[0]
                            mask[freq_slice, :] = True  
                        elif len(cluster) == 2:  
                            mask[cluster[0], cluster[1]] = True
                        cluster_mask = np.logical_or(cluster_mask, mask)


        if clusters:
            for i, cluster in enumerate(clusters):
                if cluster_p_values[i] <= 0.05:
                    if isinstance(cluster, np.ndarray):
                        cluster_indices = np.where(cluster)
                    else:  
                        cluster_indices = cluster

                    if len(cluster_indices) == 1:  
                        freq_idx = np.arange(data1.shape[0])[cluster_indices[0]]
                        time_idx = np.arange(data1.shape[1])  
                    else:  
                        freq_idx = cluster_indices[0]
                        time_idx = cluster_indices[1]
                    time_points = time_idx * (1 / tfr_ev1.info["sfreq"])
                    freq_points = freq_idx * (tfr_ev1.freqs[1] - tfr_ev1.freqs[0])
                    result = {
                        'event1': event1,
                        'event2': event2,
                        'channel': tfr.ch_names[ch_idx],
                        'cluster_p_value': cluster_p_values[i],
                        'cluster_time_points': time_points,
                        'cluster_freq_points': freq_points
                    }
                    results.append(result)


        ax_mask.imshow(cluster_mask, cmap='gray', aspect='auto', extent=[-1, 3, 0, 43])
        ax_mask.set_title(f"Mask - {tfr.ch_names[ch_idx]}", fontsize=10)
        ax_mask.axvline(0, linewidth=1, color="black", linestyle=":")
        if idx != 0:
            ax_mask.set_yticklabels("")
            ax_mask.set_ylabel("")


        diff_tfr = tfr_ev1.copy()
        diff_tfr.data = tfr_ev1.data - tfr_ev2.data
        diff_tfr.plot(
            picks=[ch_idx],
            cmap="RdBu_r",
            axes=ax_diff,
            colorbar=False,
            show=False,
            mask=cluster_mask,
            mask_style="mask"
        )
        vlim = abs(diff_tfr.data[ch_idx, :, :]).max()
        ax_diff.images[-1].set_clim(-vlim, vlim)  

        ax_diff.set_title(f"{tfr.ch_names[ch_idx]}", fontsize=10)
        ax_diff.axvline(0, linewidth=1, color="black", linestyle=":")
        if idx != 0:
            ax_diff.set_ylabel("")
            ax_diff.set_yticklabels("")


    fig.colorbar(axes[1, 0].images[-1], cax=axes[1, -1]).ax.set_yscale("linear")
    fig.suptitle(f"TFR Difference and Masks: {event1} vs {event2}\nNo baseline correction applied")
    plt.show()

results_df = pd.DataFrame(results)
print(results_df)

mask = (results_df['cluster_freq_points'].str.len() != 0)
df1 = results_df.loc[mask]
df1